# PRE_16 - Transformacion BI

Primera etapa del pipeline de BI: pasar de tres tablas operativas (clientes, productos, lineas de venta) a **una tabla analitica desnormalizada** lista para analizar.

- Grano: una fila por linea de venta (`order_id`, `line_id`).
- Se agregan atributos de cliente, producto y fecha, y metricas derivadas.
- Antes de transformar se valida integridad referencial y tipos; asi un dato malo falla aqui y no en el dashboard.

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

# Raiz de la actividad: funciona desde la carpeta (tests) o desde notebooks/ (Jupyter)
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA, OUT = ROOT / "data", ROOT / "submission"
OUT.mkdir(exist_ok=True)

## 1. Carga y validacion
Llaves unicas, sin huerfanos (cada linea apunta a un cliente y producto existentes), sin nulos, cantidades positivas y descuento en [0, 1].

In [2]:
customers = pd.read_csv(DATA / "customers.csv")
products = pd.read_csv(DATA / "products.csv")
lines = pd.read_csv(DATA / "order_lines.csv", parse_dates=["order_date"])
print(len(customers), "clientes;", len(products), "productos;", len(lines), "lineas")

# Validaciones de entrada: llaves unicas, integridad referencial, dominios y tipos
assert customers.customer_id.is_unique and products.product_id.is_unique
assert not lines.duplicated(["order_id", "line_id"]).any(), "grano (order_id, line_id) repetido"
assert lines.customer_id.isin(customers.customer_id).all(), "cliente huerfano"
assert lines.product_id.isin(products.product_id).all(), "producto huerfano"
assert lines.notna().all().all() and customers.notna().all().all() and products.notna().all().all()
assert (lines.quantity > 0).all() and lines.discount_pct.between(0, 1).all()
assert (products.unit_price > 0).all()
print("Entradas validas")

60 clientes; 12 productos; 471 lineas
Entradas validas


## 2. Transformacion
Reglas de negocio (deducidas del mart de referencia `sales_mart.db`):

- `gross_sales = quantity * unit_price`
- `discount_amount = gross_sales * discount_pct`
- `net_sales = gross_sales - discount_amount = gross_sales * (1 - discount_pct)`

Tambien se derivan `year`, `month` y `quarter` desde la fecha.

In [3]:
df = (lines.merge(customers, on="customer_id", how="left", validate="many_to_one")
           .merge(products, on="product_id", how="left", validate="many_to_one"))
assert len(df) == len(lines), "el join duplico o perdio filas"

df["year"] = df.order_date.dt.year
df["month"] = df.order_date.dt.month
df["quarter"] = df.order_date.dt.quarter
df["gross_sales"] = df.quantity * df.unit_price
df["discount_amount"] = (df.gross_sales * df.discount_pct).round(2)
df["net_sales"] = (df.gross_sales - df.discount_amount).round(2)
df["order_date"] = df.order_date.dt.strftime("%Y-%m-%d")

cols = ["order_id", "line_id", "order_date", "year", "quarter", "month",
        "customer_id", "region", "segment", "product_id", "product", "category",
        "unit_price", "quantity", "discount_pct", "gross_sales", "discount_amount", "net_sales"]
sales = df[cols].sort_values(["order_id", "line_id"]).reset_index(drop=True)
sales.head()

,order_id,line_id,order_date,year,quarter,month,customer_id,region,segment,product_id,product,category,unit_price,quantity,discount_pct,gross_sales,discount_amount,net_sales
0,1,1,2024-01-04,2024,1,1,51,Sur,Hogar,12,Producto 12,Servicios,490,5,0.05,2450,122.5,2327.5
1,1,2,2024-01-04,2024,1,1,51,Sur,Hogar,1,Producto 1,Tecnología,780,4,0.10,3120,312.0,2808.0
2,1,3,2024-01-04,2024,1,1,51,Sur,Hogar,3,Producto 3,Tecnología,510,2,0.00,1020,0.0,1020.0
3,2,1,2024-01-07,2024,1,1,35,Centro,Pyme,7,Producto 7,Oficina,240,5,0.05,1200,60.0,1140.0
4,3,1,2024-01-10,2024,1,1,16,Norte,Corporativo,6,Producto 6,Oficina,290,5,0.10,1450,145.0,1305.0


## 3. Cuadre contra el mart de referencia
La tabla debe reproducir los totales del data mart ya construido (hechos y ventas netas).

In [4]:
con = sqlite3.connect(f"file:{DATA / 'sales_mart.db'}?mode=ro", uri=True)
mart = pd.read_sql("select order_id, line_id, gross_sales, net_sales from fact_sales", con)
con.close()

chk = sales.merge(mart, on=["order_id", "line_id"], suffixes=("", "_mart"), validate="one_to_one")
assert len(chk) == len(mart) == len(sales)
assert (chk.gross_sales == chk.gross_sales_mart).all()
assert ((chk.net_sales - chk.net_sales_mart).abs() < 0.01).all()
assert abs(sales.gross_sales.sum() - sales.net_sales.sum() - sales.discount_amount.sum()) < 0.01
print("Cuadra con el mart: bruto", sales.gross_sales.sum(), "| neto", round(sales.net_sales.sum(), 2))

Cuadra con el mart: bruto 694560 | neto 672990.5


## 4. Entregable
Se escribe `submission/sales_analytics.csv` y se relee para confirmar tipos y numero de filas.

In [5]:
sales.to_csv(OUT / "sales_analytics.csv", index=False)
back = pd.read_csv(OUT / "sales_analytics.csv", parse_dates=["order_date"])
assert len(back) == 471 and back.notna().all().all()
print(back.shape); back.dtypes

(471, 18)


order_id                    int64
line_id                     int64
order_date         datetime64[ns]
year                        int64
quarter                     int64
month                       int64
customer_id                 int64
region                     object
segment                    object
product_id                  int64
product                    object
category                   object
unit_price                  int64
quantity                    int64
discount_pct              float64
gross_sales                 int64
discount_amount           float64
net_sales                 float64
dtype: object